In [2]:
import numpy as np 
import heapq as hq
import collections as col

In [3]:
a = [1,2,3]

a[: :-1]

[3, 2, 1]

### Tableau implementation Class

In [4]:
# class that implements tableau and necessary operations
# Reference -Improved Simulation of Stabilizer Circuits by Scott Aaronson & Daniel Gottesman
class Cirq_Tableau:
    
    def __init__(
        self,
        pauli_word: list[str] = None  
    ):
        if pauli_word is None or len(pauli_word) == 0:
            self._column_num = None
            self._row_num = None

            self._ss, self._xs, self._zs = None, None, None
        else:
            if "-" in pauli_word[0]:
                self._column_num = len(pauli_word[0][1:]) 
            else:
                self._column_num = len(pauli_word[0])
            self._row_num = len(pauli_word)

            self._ss = self.create_sign(pauli_word)
            self._xs, self._zs = self.create_tab(pauli_word)

    # setters & getters 
    @property
    def ss(self) -> np.ndarray:
        return self._ss

    @ss.setter 
    def ss(self, new_ss: np.ndarray):
        self._ss = new_ss
        
    @property
    def xs(self) -> np.ndarray:
        return self._xs

    @xs.setter 
    def xs(self, new_xs: np.ndarray):
        self._xs = new_xs
        
    @property
    def zs(self) -> np.ndarray:
        return self._zs

    @zs.setter 
    def zs(self, new_zs: np.ndarray):
        self._zs = new_zs
        
    @property
    def column_num(self) -> int:
        return self._column_num

    @column_num.setter 
    def column_num(self, new_num: int):
        self._column_num = new_num
    
    @property
    def row_num(self) -> int:
        return self._row_num

    @row_num.setter 
    def row_num(self, new_num: int):
        self._row_num = new_num

    # functions that create parts of tableau
    def create_sign(self, pauli_word: list[str]):
        temp_ss = np.zeros((self.row_num), dtype=int)
        for pauli_string in pauli_word:
            if "-" in pauli_string:
                index = pauli_word.index(pauli_string)
                pauli_word[index] = pauli_string.replace("-", "")
                #print(pauli_word)
                temp_ss[index] = 1
        return temp_ss
        
    def create_tab(self, pauli_word: list[str]):
        temp_x = np.zeros((self.row_num, self.column_num), dtype=int)
        temp_z = np.zeros((self.row_num, self.column_num), dtype=int)
        for i, pauli_string in enumerate(pauli_word):
            for j, pauli in enumerate(pauli_string):
                if pauli == "X" or pauli == "Y":
                    temp_x[i][j] = 1
                if pauli == "Z" or pauli == "Y":
                    temp_z[i][j] = 1
        return temp_x, temp_z

    # Clifford operations on tableau. 
    def apply_H(self,column: int):
        self.ss ^= self.xs[:, column] & self.zs[:, column]
        self.xs[:, column], self.zs[:, column] = self.zs[:, column].copy(), self.xs[:, column].copy()

    def apply_S(self, column: int):
        self.ss ^= self.xs[:, column] & self.zs[:, column]
        self.zs[:, column] = self.xs[:, column] ^ self.zs[:, column]

    def apply_CX(self, control: int, target: int):
        self.ss ^= (
            (self.xs[:, control] & self.zs[:, target])
            &(~(self.xs[:, target] ^ self.zs[:, control]))
        )
        self.xs[:, target] ^= self.xs[:, control]
        self.zs[:, control] ^= self.zs[:, target]

    # class operations necessary for comparisons and equating
    def copy(self):
        new_tab = Cirq_Tableau()
        new_tab.column_num = self.column_num
        new_tab.row_num = self.row_num
        new_tab.ss = self.ss.copy()
        new_tab.zs = self.zs.copy()
        new_tab.xs = self.xs.copy()
        return new_tab
    
    def __eq__(self, other):
        if not isinstance(other, type(self)):
            return NotImplemented  
        return (
            self.column_num == other.column_num
            and self.row_num == other.row_num
            and np.array_equal(self.ss, other.ss)
            and np.array_equal(self.xs, other.xs)
            and np.array_equal(self.zs, other.zs)
        )
    
    def return_string(self):
        string = ''
        for i in range(self.row_num):
            if self.ss[i]:
                string += "-"  

            for j in range(self.column_num):
                if self.xs[i][j] and not self.zs[i][j]:
                    string += "X"
                elif not self.xs[i][j] and self.zs[i][j]:
                    string += "Z"
                elif self.xs[i][j] and self.zs[i][j]:
                    string += "Y"
                else:
                    string += "I"
            if i < self.row_num - 1:
                string += "\n" 

        return string
    
    def __copy__(self):
        return self.copy()
        

    def __str__(self) -> str:
        ss = np.expand_dims(self.ss, axis = 1)
        xz = np.concatenate((self.xs, self.zs, ss), axis=1)
        return str(xz)

    def __hash__(self) -> int:
        return hash(self.zs.tobytes() + self.xs.tobytes() + self.ss.tobytes())

    def __lt__(self, other):
        return self.row_num < other.row_num

### Functions

In [5]:
def edges(num_qubits: int):
    '''
    Function to create list of action tuples for each state 
    action tuples -> (operation: str, qubits acted on: int/s, weight of action: int)
    
    :param num_qubits: Integer that represents number of qubits 
    '''
    container = [] # list to hold all elements

    # TODO: make creation of possible actions less crude than listing out every combination - not good but works for now
    # loop to create action tuples for every possible combination
    for i in range(num_qubits):
        if i != (num_qubits-1):
            for j in range((i+1),num_qubits):
                container.append([("CX", i, j, 2)])
                container.append([("S", i, 1),("CX", i, j, 2)])
                container.append([("H", i, 1),("CX", i, j, 2)])
                container.append([("S", j, 1),("CX", i, j, 2)])
                container.append([("H", j, 1),("CX", i, j, 2)])
                container.append([("S", j, 1),("S", i, 1),("CX", i, j, 2)])
                container.append([("H", j, 1),("S", i, 1),("CX", i, j, 2)])
                container.append([("S", j, 1),("H", i, 1),("CX", i, j, 2)])
                container.append([("H", j, 1),("H", i, 1),("CX", i, j, 2)])
                container.append([("CX", j, i, 2)])
                container.append([("S", i, 1),("CX", j, i, 2)])
                container.append([("H", i, 1),("CX", j, i, 2)])
                container.append([("S", j, 1),("CX", j, i, 2)])
                container.append([("H", j, 1),("CX", j, i, 2)])
                container.append([("S", j, 1),("S", i, 1),("CX", j, i, 2)])
                container.append([("H", j, 1),("S", i, 1),("CX", j, i, 2)])
                container.append([("S", j, 1),("H", i, 1),("CX", j, i, 2)])
                container.append([("H", j, 1),("H", i, 1),("CX", j, i, 2)])
                
    return container

In [6]:
def prune(tableau: Cirq_Tableau):
    '''
    Function to remove rows in a tableau with only a single nonidentity operation
    
    :param tableau: Cirq_Tableau to act on
    '''
    prn_ndxs = [] # list to hold all row indices to be pruned
    
    tab = tableau.copy() # copy of tableau to work on

    single_q = [] # loist of single qubit operations 
    
    # extracts x, z and sign arrays
    x, z, s = tab.xs, tab.zs, tab.ss

    # bitwise ORs x and z arrays in tableau to create array where nonidentity operation indices have 1 in them
    weight_array = x | z

    # loop that checks through each row to find out if it has a Pauli weight of 1
    for r_ndx in range(len(weight_array)):
        if sum(weight_array[r_ndx])  == 1:

            # appends index to be pruned if Pauli weight == 1
            prn_ndxs.append(r_ndx)

            # checks what type of Pauli is on that index X, Y or Z and appends to path with action weight of zero 
            if sum(z[r_ndx]) == 0:
                if s[r_ndx] == 0:
                    single_q.append([("X", int(np.argmax(x[r_ndx] == 1)), 0)])
                else:
                    single_q.append([("-X", int(np.argmax(x[r_ndx] == 1)), 0)])
            elif sum(x[r_ndx]) == 0:
                if s[r_ndx] == 0:
                    single_q.append([("Z", int(np.argmax(z[r_ndx] == 1)), 0)])
                else:
                    single_q.append([("-Z", int(np.argmax(z[r_ndx] == 1)), 0)])
            else:
                if s[r_ndx] == 0:
                    single_q.append([("Y", int(np.argmax(x[r_ndx] == 1)), 0)])
                else:
                    single_q.append([("-Y", int(np.argmax(x[r_ndx] == 1)), 0)])
                    
        elif sum(weight_array[r_ndx])  == 0: # elif removes fully I lines 
            x, z, s = np.delete(x, r_ndx, axis=0), np.delete(z, r_ndx, axis=0), np.delete(s, r_ndx)
            
    # prunes rows 
    x, z, s = np.delete(x, prn_ndxs, axis=0), np.delete(z, prn_ndxs, axis=0), np.delete(s, prn_ndxs)
    column_num = len(x[0]) if x.size != 0 else 0 
    row_num = len(x)

    # updates tableau and returns it 
    tab.xs, tab.zs, tab.ss, tab.column_num, tab.row_num = x, z, s, column_num,row_num
    return tab, single_q
    

In [7]:
def heuristic(tab: Cirq_Tableau):
    '''
    Function to approximate "distance" to state with only single qubit Pauli strings

    :param tab: Cirq_Tableau used to calculate value
    '''

    # bitwise ORs x and z arrays in tableau to create array where nonidentity operation indices| have 1 in them
    compress = tab.xs | tab.zs

    # returns sum of the sum of each row less one
    return sum([int(sum(line))-1 for line in compress])

In [8]:
def reconstruct(parents, current_state):
    '''
    Function to reconstruct the path to the goal state

    ;param parents: Dictionary of states as keys and previous states and operations as parents 
    :param current_state: Cirq_Tableau for current state
    '''

    current = current_state
    path = col.deque()
    while current in parents:
        path.extendleft(parents[current][1][: : -1])
        path.extendleft(parents[current][2][: : -1])
        current = parents[current][0]

    return list(path)

In [9]:
def A_star_prune(start: Cirq_Tableau, h):
    open_list = [] # priority queue used to store nodes to visit
    hq.heappush(open_list, (0, start)) # push the start into the priority queue - acts as a min-heap

    parents = dict() # dictionary to store the parent states of each state and the associated operation to get to it
    g_value = dict({start: 0}) # dictionary to store the distance to a current state


                   
    while open_list:
        
        # acquires the current state with lowest f-value 
        current = hq.heappop(open_list) 
        current_state = current[1]

        # reconstruct the path that led to the empty state
        if current_state.row_num == 0:
            return reconstruct(parents, current_state) 

        current_edges = edges(current_state.column_num)

        for chunk in current_edges:
            tab = current_state.copy()
            for op in chunk:
                match op[0]:
                    case "CX":
                        tab.apply_CX(op[1], op[2])
                    case "H":
                        tab.apply_H(op[1])
                    case "S":
                        tab.apply_S(op[1])
            
            tab, single_q = prune(tab)
            new_g_value = g_value[current_state] + sum([x[-1] for x in chunk])
            new_f_value = new_g_value + h(tab)
            
            if tab in g_value:
                if new_g_value < g_value[tab]:
                    parents[tab] = [current_state, single_q, chunk]
                    g_value[tab] = new_g_value
                    hq.heappush(open_list, (new_f_value, tab))
                    
            else:
                parents.update({tab : [current_state,single_q, chunk]})
                g_value.update({tab : new_g_value})
                hq.heappush(open_list, (new_f_value, tab))
        

In [10]:
# dumby Pauli Word to test how search prune works - created by Mulundano  
word = ["-XXII", "IIZZ"]

In [11]:
tableau = Cirq_Tableau(word)
print(tableau)

[[1 1 0 0 0 0 0 0 0 0 1]
 [0 0 0 0 0 0 0 1 1 0 0]]


In [40]:
path = A_star_prune(tableau, heuristic)
path

[('CX', 0, 1, 2), [('X', 0, 0)], ('CX', 2, 3, 2), [('Z', 3, 0)]]